### Assignment 3
### George Boufis - f3352504


### Loading the Web Graph and Constructing the Transition Matrix

In this step, we load the Stanford web graph dataset and construct the transition probability matrix $P$.

Each row of the dataset represents a directed edge between two pages, along with the corresponding transition probability. Using this data, we build a sparse matrix $P \in \mathbb{R}^{n \times n}$, where $n$ is the total number of pages.

The matrix $P$ is constructed so that:
- each row corresponds to a web page,
- each entry $P_{ij}$ represents the probability of moving from page $i$ to page $j$,
- the rows of $P$ sum to 1, making it a valid stochastic matrix.

We use a sparse representation (CSR format) to efficiently handle the large size of the graph.

In [1]:
import numpy as np
from scipy.sparse import coo_matrix

# Load the .dat file
data = np.loadtxt("/Users/giorgosboufis/Desktop/Master Data Science and AI/2nd Semester/Numerical Optimization/Assignments/Assignment 3/stanweb.dat")

# Split columns
rows = data[:, 0].astype(int)
cols = data[:, 1].astype(int)
vals = data[:, 2]

print("Loaded shape:", data.shape)
print("First 5 rows:")
print(data[:5])

# Check node numbering
print("Minimum node id:", min(rows.min(), cols.min()))
print("Maximum node id:", max(rows.max(), cols.max()))

# Number of nodes
n = max(rows.max(), cols.max())
print("Number of nodes n =", n)

# Convert to 0-based indexing for Python
rows0 = rows - 1
cols0 = cols - 1

# Build sparse matrix P
P = coo_matrix((vals, (rows0, cols0)), shape=(n, n)).tocsr()

print("Sparse matrix shape:", P.shape)
print("Number of nonzeros:", P.nnz)

# Check some row sums
row_sums = np.array(P.sum(axis=1)).flatten()
print("First 10 row sums:", row_sums[:10])




Loaded shape: (2382912, 3)
First 5 rows:
[[1.00000000e+00 6.54800000e+03 5.00000000e-01]
 [1.00000000e+00 1.54090000e+04 5.00000000e-01]
 [2.00000000e+00 2.52915000e+05 3.22580645e-02]
 [2.00000000e+00 2.46897000e+05 3.22580645e-02]
 [2.00000000e+00 2.51658000e+05 3.22580645e-02]]
Minimum node id: 1
Maximum node id: 281903
Number of nodes n = 281903
Sparse matrix shape: (281903, 281903)
Number of nonzeros: 2312497
First 10 row sums: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Handling Dangling Nodes

In this step, we identify dangling nodes, i.e., pages with no outgoing links.

We construct a vector $a \in \mathbb{R}^n$ such that:

$$
a_i =
\begin{cases}
1 & \text{if page } i \text{ has no out-links} \\
0 & \text{otherwise}
\end{cases}
$$

This vector is used later to redistribute the PageRank mass of dangling nodes uniformly across all pages.

In [2]:
row_sums = np.array(P.sum(axis=1)).flatten()
a = (row_sums == 0).astype(float)

print("Dangling nodes:", int(a.sum()))

Dangling nodes: 172



### Power Method for PageRank ($\alpha = 0.85$)

In this step, we compute the PageRank vector using the Power Method with damping factor $\alpha = 0.85$.

We use the uniform teleportation vector

$$
v = \frac{1}{n}e
$$

and initialize the iteration with the uniform probability vector.

At each iteration, the PageRank vector is updated according to:

$$
x^{(k+1)} = \alpha x^{(k)} P + \left( \alpha (x^{(k)} a) + (1-\alpha) \right) v
$$

where:
- $P$ is the transition matrix,
- $a$ is the dangling-node indicator vector,
- $x^{(k)} a$ is the total PageRank mass of dangling nodes.

The iteration stops when the $L^1$ difference between two consecutive iterates becomes smaller than

$$
\tau = 10^{-8}
$$

This gives the PageRank vector for the Stanford web graph using the Power Method.

In [3]:
import time 
alpha = 0.85
tau = 1e-8
max_iter = 1000

v = np.ones(n) / n
x = np.ones(n) / n

start = time.time()

for k in range(max_iter):
    dangling_mass = x @ a
    x_new = alpha * (x @ P) + (alpha * dangling_mass + (1 - alpha)) * v

    err = np.linalg.norm(x_new - x, 1)

    if k % 10 == 0:
        print(f"Iteration {k} | Error = {err:.2e}")

    if err < tau:
        x = x_new
        print(f"\nConverged in {k+1} iterations")
        break

    x = x_new

end = time.time()

print("Time:", end - start)
print("Sum:", x.sum())

Iteration 0 | Error = 9.58e-01
Iteration 10 | Error = 1.06e-02
Iteration 20 | Error = 1.37e-03
Iteration 30 | Error = 2.23e-04
Iteration 40 | Error = 3.88e-05
Iteration 50 | Error = 7.00e-06
Iteration 60 | Error = 1.29e-06
Iteration 70 | Error = 2.41e-07
Iteration 80 | Error = 4.54e-08
Iteration 90 | Error = 8.61e-09

Converged in 91 iterations
Time: 0.9390180110931396
Sum: 0.9999999999999996


### Top 10 Pages According to PageRank

After computing the PageRank vector with the Power Method, we extract the 10 pages with the highest PageRank values.

This helps us inspect the most important nodes in the Stanford web graph and later compare the ranking obtained by different solution methods.

In [4]:
top10_idx = np.argsort(-x)[:10]

print("Top 10 nodes by PageRank:")
for i, idx in enumerate(top10_idx, start=1):
    print(f"{i}. Node {idx+1} : PR = {x[idx]:.12e}")

Top 10 nodes by PageRank:
1. Node 89073 : PR = 1.130283590952e-02
2. Node 226411 : PR = 9.287655722181e-03
3. Node 241454 : PR = 8.297234856409e-03
4. Node 262860 : PR = 3.023116558529e-03
5. Node 134832 : PR = 3.001266418801e-03
6. Node 234704 : PR = 2.572256254068e-03
7. Node 136821 : PR = 2.453702650270e-03
8. Node 68889 : PR = 2.430779705909e-03
9. Node 105607 : PR = 2.397427531186e-03
10. Node 69358 : PR = 2.364003605430e-03


### Linear System Formulation for PageRank

We reformulate the PageRank equation as a linear system.

Starting from

$$
x = \alpha P^T x + b
$$

we obtain:

$$
(I - \alpha P^T)x = b
$$

Thus, we define the matrix

$$
A = I - \alpha P^T
$$

which will be used to solve the PageRank problem using an iterative linear solver (Gauss–Seidel method).

The matrix is stored in sparse format due to its large size.

In [5]:
from scipy.sparse import identity, tril, triu 

alpha = 0.85
tau = 1e-8
max_iter = 1000

v = np.ones(n) / n 

A = identity(n, format="csr") - alpha * P.transpose().tocsr()

print("A shape:", A.shape)
print("A nnz:", A.nnz)

A shape: (281903, 281903)
A nnz: 2594400


### Diagonal of Matrix $A$

We examine the diagonal entries of the matrix

$$
A = I - \alpha P^T
$$

The results show that all diagonal entries are equal to 1.

This is important because:
- It guarantees that $A_{ii} \neq 0$ for all $i$,
- Therefore, the Gauss–Seidel method is well-defined,
- It also contributes to the numerical stability and convergence of the iterative method.

In [6]:
diag_A = A.diagonal()

print("Min diagonal entry:", diag_A.min())
print("Max diagonal entry:", diag_A.max())

Min diagonal entry: 1.0
Max diagonal entry: 1.0


### Gauss–Seidel Method for the Linear-System Formulation

The PageRank vector satisfies:

$$
x = \alpha P^T x + \alpha (x^T a)\, v + (1-\alpha)v
$$

which can be written as:

$$
(I - \alpha P^T)x = (\alpha (x^T a) + (1-\alpha))v
$$

In practice, we solve a simplified linear system of the form:

$$
(I - \alpha P^T)x = v
$$

and normalize the solution so that its entries sum to 1.

This approach is valid because the PageRank vector is defined up to scaling, and normalization recovers the correct probability distribution.

In [7]:
x_gs = np.ones(n) / n   # initial guess

start = time.time()

for k in range(max_iter):
    x_old = x_gs.copy()

    for i in range(n):
        row_start = A.indptr[i]
        row_end = A.indptr[i + 1]

        cols_i = A.indices[row_start:row_end]
        vals_i = A.data[row_start:row_end]

        s = 0.0
        aii = 0.0

        for j, aij in zip(cols_i, vals_i):
            if j == i:
                aii = aij
            else:
                s += aij * x_gs[j]   # uses updated values when available

        x_gs[i] = (v[i] - s) / aii

    err = np.linalg.norm(x_gs - x_old, 1)

    if k % 10 == 0:
        print(f"Iteration {k} | Error = {err:.2e}")

    if err < tau:
        print(f"\nGauss-Seidel converged in {k+1} iterations")
        break

end = time.time()

# normalize to sum to 1
x_gs = x_gs / x_gs.sum()

print("Time:", end - start)
print("Sum after normalization:", x_gs.sum())

Iteration 0 | Error = 1.31e+00
Iteration 10 | Error = 6.78e-02
Iteration 20 | Error = 2.82e-03
Iteration 30 | Error = 1.19e-04
Iteration 40 | Error = 5.11e-06
Iteration 50 | Error = 2.24e-07
Iteration 60 | Error = 1.01e-08

Gauss-Seidel converged in 62 iterations
Time: 61.567142963409424
Sum after normalization: 1.0000000000000002


### Comparing the Two Methods

To verify whether the Power Method and the Gauss–Seidel method produce the same PageRank vector, we compute the $L^1$ distance between the two final solutions:

$$
\|x_{\mathrm{GS}} - x_{\mathrm{PM}}\|_1
$$

A very small value indicates that the two methods converge to essentially the same PageRank vector.

In [8]:
diff = np.linalg.norm(x_gs - x, 1)
print(" Difference between Power Method and Gauss-Seidel:", diff)

 Difference between Power Method and Gauss-Seidel: 1.0826482274346026e-08


### Top 10 Pages from the Gauss–Seidel Solution

We extract the 10 pages with the highest PageRank values from the Gauss–Seidel solution.

Comparing these nodes with the ranking obtained from the Power Method allows us to verify that both methods produce essentially the same ordering of the most important pages.

In [9]:
top10_idx_gs = np.argsort(-x_gs)[:10]

print("Top 10 nodes by Gauss-Seidel:")
for i, idx in enumerate(top10_idx_gs, start=1):
    print(f"{i}. Node {idx+1} :  PR = {x_gs[idx]:.12e}")

Top 10 nodes by Gauss-Seidel:
1. Node 89073 :  PR = 1.130283580990e-02
2. Node 226411 :  PR = 9.287655699655e-03
3. Node 241454 :  PR = 8.297234869131e-03
4. Node 262860 :  PR = 3.023116515598e-03
5. Node 134832 :  PR = 3.001266422155e-03
6. Node 234704 :  PR = 2.572256249695e-03
7. Node 136821 :  PR = 2.453702654296e-03
8. Node 68889 :  PR = 2.430779709460e-03
9. Node 105607 :  PR = 2.397427529357e-03
10. Node 69358 :  PR = 2.364003612134e-03


### Final Conclusion

Both the Power Method and the Gauss–Seidel method converge to essentially the same PageRank vector, as confirmed by the very small $L^1$ difference between the two solutions and the identical ranking of the top nodes.

Although the Gauss–Seidel method requires fewer iterations to converge, it is significantly slower in execution time due to its sequential updates and lack of efficient vectorization.

Therefore, for large-scale problems such as the Stanford web graph, the Power Method is preferred in practice due to its computational efficiency.

### Question b)

### Power Method for PageRank ($\alpha = 0.99$)

We repeat the Power Method using a larger damping factor,

$$
\alpha = 0.99
$$

in order to study how the convergence speed and the ranking change compared with the case $\alpha = 0.85$.

The iteration is again given by:

$$
x^{(k+1)} = \alpha x^{(k)} P + \left( \alpha (x^{(k)} a) + (1-\alpha) \right) v
$$

with stopping criterion

$$
\|x^{(k+1)} - x^{(k)}\|_1 < 10^{-8}.
$$

We store the result in a separate vector so that we can later compare the ranking for $\alpha = 0.99$ with the ranking obtained for $\alpha = 0.85$.

In [10]:
import time

alpha = 0.99
tau = 1e-8
max_iter = 2500

v = np.ones(n) / n
x_099 = np.ones(n) / n

start = time.time()
converged = False

for k in range(max_iter):
    dangling_mass = x_099 @ a
    x_new = alpha * (x_099 @ P) + (alpha * dangling_mass + (1 - alpha)) * v

    err = np.linalg.norm(x_new - x_099, 1)

    if k % 100 == 0:
        print(f"Iteration {k} | Error = {err:.2e}")

    if err < tau:
        x_099 = x_new
        converged = True
        print(f"\nConverged in {k+1} iterations")
        break

    x_099 = x_new

end = time.time()

if not converged:
    print(f"\nDid not converge within {max_iter} iterations")

print("Time:", end - start)
print("Sum:", x_099.sum())

Iteration 0 | Error = 1.12e+00
Iteration 100 | Error = 8.00e-03
Iteration 200 | Error = 2.43e-03
Iteration 300 | Error = 8.02e-04
Iteration 400 | Error = 2.73e-04
Iteration 500 | Error = 9.45e-05
Iteration 600 | Error = 3.31e-05
Iteration 700 | Error = 1.17e-05
Iteration 800 | Error = 4.15e-06
Iteration 900 | Error = 1.48e-06
Iteration 1000 | Error = 5.32e-07
Iteration 1100 | Error = 1.92e-07
Iteration 1200 | Error = 6.93e-08
Iteration 1300 | Error = 2.51e-08

Converged in 1392 iterations
Time: 11.689427137374878
Sum: 0.9999999999999959


### Compare the first 50 nodes with a=0.85 and a=0.99

We observe from the results the following:
1. Same set of top 50 nodes? False.
2. Same exact order? False.
3. First difference at rank 12: -) a=0.85: node 95163
                                -) a=0.99: node 251796
So, the ranking of the first 50 nodes changed when a increased from 0.85 to 0.99.

In [11]:
top50_085 = np.argsort(-x)[:50]
top50_099 = np.argsort(-x_099)[:50]

print("Same set of top 50 nodes?", set(top50_085) == set(top50_099))
print("Same exact order?", np.array_equal(top50_085, top50_099))

Same set of top 50 nodes? False
Same exact order? False


The Gauss-Seidel method could also be applied for a=0.99. However, similarly to the Power Method, the convergence is expected to be significantly slower as a approches 1. Since both methods solve the same PageRank system, they lead to the same solution.

### Question c) 
The components of 𝜋 do not converge at the same speed. In our experiment with the Power Method, the components corresponding to low-ranked nodes reached much smaller absolute error after the same number of iterations than the components corresponding to highly ranked nodes. Therefore, in terms of absolute error, non-important nodes converged faster. The same non-uniform componentwise convergence is expected when solving the corresponding linear system with Gauss–Seidel, although Gauss–Seidel converges faster overall.

In [12]:
# pick nodes
top_nodes = np.argsort(-x)[:5]
bottom_nodes = np.argsort(x)[:5]

print("Top nodes:", top_nodes + 1)
print("Bottom nodes:", bottom_nodes + 1)

# track convergence
x_track = np.ones(n) / n
history = {node: [] for node in np.concatenate([top_nodes, bottom_nodes])}

for k in range(100):
    for node in history:
        history[node].append(x_track[node])

    dangling_mass = x_track @ a
    x_new = 0.85 * (x_track @ P) + (0.85 * dangling_mass + (1 - 0.85)) * (np.ones(n)/n)

    x_track = x_new

# compare final error per node
for node in history:
    err = abs(history[node][-1] - x[node])
    print(f"Node {node+1} error: {err:.3e}")

Top nodes: [ 89073 226411 241454 262860 134832]
Bottom nodes: [     1 100721 216231 216228  23318]
Node 89073 error: 8.969e-11
Node 226411 error: 3.191e-11
Node 241454 error: 1.013e-11
Node 262860 error: 3.549e-11
Node 134832 error: 3.981e-12
Node 1 error: 4.656e-18
Node 100721 error: 4.656e-18
Node 216231 error: 4.656e-18
Node 216228 error: 4.656e-18
Node 23318 error: 4.656e-18


### Question a) 

Let the original PageRank vector be: 

$$
\pi = (\pi_1, \pi_2, \dots, \pi_n), \quad \sum_{i=1}^{n} \pi_i = 1
$$

After adding the new page $X$, the PageRank vector becomes:

$$
(\tilde{\pi}_1, \tilde{\pi}_2, \dots, \tilde{\pi}_n, x)
$$

Using the stationary equations of PageRank, we express the new PageRanks in terms of the original vector $\pi$.

For the old pages, we obtain: 

$$
\tilde{\pi} = \frac{n}{n+1} \pi
$$

which means that for each page $i = 1, \dots, n$:

$$
\tilde{\pi}_i = \frac{n}{n+1} \pi_i
$$

For the new page $X$, we have:

$$
x = \frac{1}{n+1}
$$

### Interpretation

All old pages lose the same multiplicative factor:

$$
\frac{n}{n+1}
$$

Since $n$ is very large:

$$
\frac{n}{n+1} \approx 1
$$

Thus, the decrease in PageRank for each old page is very small, and the relative ranking among the old pages remains unchanged.



### Question b) 



Now we add a second new page $Y$, with no in-links, and with one out-link pointing to $X$.

Let the new PageRank vector be:

$$
(\hat{\pi}_1, \hat{\pi}_2, \dots, \hat{\pi}_n, x, y)
$$

where: 
- $\hat{\pi}$ denotes the PageRanks of the old $n$ pages,
- $x$ is the PageRank of page $X$,
- $y$ is the PageRank of page $Y$.

For the old pages, the teleportation mass in now distributed over $n+2$ pages, so as in part (a) we obtain:

$$
\hat{\pi} = \frac{n}{n+2}\pi
$$

Page $Y$ has no in-links, so it receives only teleportation:

$$
y = \frac{1-\alpha}{n+2}
$$

Page $X$ receives PageRank from:
- itself, since it has no out-links and therefore gets a self-loop in $P$,
- page $Y$, which points to $X$,
- teleportation

Thus its stationary equation is :



$$
x = \alpha x + \alpha y + \frac{1-\alpha}{n+2}
$$


Substituting $y = \dfrac{1-\alpha}{n+2}$ gives:


$$
x = \alpha x + \alpha \frac{1-\alpha}{n+2} + \frac{1-\alpha}{n+2}
$$


$$
x - \alpha x = \frac{(1-\alpha)(1+\alpha)}{n+2}
$$


$$
(1-\alpha)x = \frac{(1-\alpha)(1+\alpha)}{n+2}
$$


$$
x = \frac{1+\alpha}{n+2}
$$


Therefore, the new PageRanks are:

$$
\hat{\pi} = \frac{n}{n+2}\pi
$$


$$
x = \frac{1+\alpha}{n+2}
$$


$$
y = \frac{1-\alpha}{n+2}
$$


### Does the PageRank of $X$ improve?

In part (a), we had:

$$
x_{(a)} = \frac{1}{n+1}
$$


Now we have:


$$
x_{(b)} = \frac{1+\alpha}{n+2}
$$


Since

$$
\frac{1+\alpha}{n+2} > \frac{1}{n+1}
$$


for large $n$ (and in particular for the usual PageRank choice $\alpha = 0.85$), the PageRank of $X$ increases.

Thus, adding page $Y$ that points to $X$ improves the PageRank of $X$.



### Question c)




To maximize the PageRank of page $X$, all links in the small link farm should be arranged so that they send as much PageRank as possible toward $X$, while preventing PageRank from leaving $X$.

The best structure is:

- page $Y$ links to $X$,
- page $Z$ links to $X$,
- page $X$ has no out-links.

Because of the PageRank convention in the assignment, a page with no out-links gets a self-loop in $P$. Therefore, if $X$ has no out-links, it effectively points to itself.

So the optimal structure is:

$$
Y \to X, \qquad Z \to X, \qquad X \to X
$$

where the self-loop of $X$ is induced automatically by the dangling-node rule.

### This is optimal because: 

If $Y$ or $Z$ pointed somewhere else, then part of their PageRank would be wasted before reaching $X$.

If $X$ linked to $Y$ or $Z$, then some of the PageRank of $X$ would leave $X$, which would reduce its value.

If $Y$ and $Z$ linked to each other, they would circulate PageRank between themselves instead of passing it directly to $X$.

Therefore, the best strategy is for both supporting pages to point directly to $X$, while $X$ keeps all the PageRank it receives by having no out-links.

### Question d) 




No, adding links from page $X$ to older, popular pages does **not** improve the PageRank of $X$.

In the optimal structure from part (c), page $X$ has no out-links, so by the rule of the assignment it becomes a dangling node and gets a self-loop in $P$. This is beneficial for $X$, because it keeps recycling its own PageRank.

If we add links from $X$ to older popular pages, then $X$ is no longer dangling, and its outgoing probability is split among several destinations. As a result, part of the PageRank of $X$ flows away from $X$, so its PageRank does not increase. In fact, it typically decreases.

The answer does not change if we add links from $Y$ or $Z$ to older, popular pages.

In the optimal structure, $Y$ and $Z$ should send all of their PageRank directly to $X$. If either of them also links to outside pages, then their outgoing probability is split, and less PageRank is transferred to $X$.

Therefore, adding links from $Y$ or $Z$ to older popular pages also does not help $X$. It generally reduces the PageRank that $X$ receives.

Thus, any extra out-link from the link farm to outside pages leaks PageRank away from $X$, and does not improve its ranking.

### Question e)



To raise the PageRank of $X$ further, I would create more supporting pages whose only purpose is to transfer PageRank to $X$.

Based on the previous parts, the best strategy is:

- create additional fake pages,
- make each of them point directly to $X$,
- keep $X$ with no out-links, so that by the rule of the assignment it gets a self-loop,
- avoid any unnecessary links among the supporting pages,
- avoid links from the farm to outside pages, since those links leak PageRank away from $X$.

The intuition is that every extra supporting page contributes PageRank to $X$, while any extra outgoing link that does not point to $X$ reduces the amount of PageRank that eventually reaches $X$.

Therefore, if a link farm has $m$ pages in total, with one target page $X$ and $m-1$ supporting pages, the optimal structure is that every supporting page points directly to $X$, and $X$ has no out-links.

So the best structure is:

$$
\text{all supporting pages} \to X
$$

with $X$ having no out-links, and therefore receiving a self-loop automatically from the dangling-node rule.

In summary, to maximize the PageRank of $X$, the farm should concentrate as much PageRank flow as possible toward $X$ and avoid any links that send PageRank elsewhere.

### Proof of the optimal link farm structure with $m$ nodes

Consider a link farm with $m$ pages in total: one target page $X$ and $m-1$ supporting pages.

We claim that the PageRank of $X$ is maximized when:

- every supporting page points only to $X$,
- $X$ has no out-links, so it gets a self-loop from the dangling-node rule,
- there are no other links inside the farm,
- there are no links from the farm to outside pages.

So the optimal structure is:

$$
Y_1 \to X,\quad Y_2 \to X,\quad \dots,\quad Y_{m-1} \to X
$$

with $X$ having no out-links.

#### Proof idea

First, $X$ should have no out-links. If $X$ has no out-links, then by the rule of the assignment it becomes a dangling node and gets a self-loop in $P$. Hence, its PageRank is retained at $X$. If $X$ had any out-links, part of its PageRank would flow away, which would reduce its value.

Second, every supporting page should point directly to $X$. A direct link transfers the whole link-based contribution of that page toward $X$. If a supporting page points somewhere else first, then part of its PageRank is delayed, split, or lost before reaching $X$.

Third, supporting pages should not link to each other. Any such link circulates PageRank among the supporting pages instead of sending it directly to $X$. Replacing such a link by a direct link to $X$ increases the flow toward $X$.

Finally, no page in the farm should link to outside pages. Any outside link leaks PageRank mass away from the farm and therefore cannot help maximize the PageRank of $X$.

Therefore, the optimal structure is a star centered at $X$, with all supporting pages pointing directly to $X$, and $X$ having no out-links.